# ⚙️ Feature Engineering

Transform raw data into model-ready features using `dskit.features`.
We build a complete preprocessing pipeline on the **Titanic dataset**.

**Outline:** Handle missing → Encode categoricals → Date features → Scale → Polynomial features → Full pipeline

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from dskit.features import (
    encode_categoricals,
    create_date_features,
    scale_features,
    handle_missing,
    add_polynomial_features,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete ✅')

In [ ]:
df_raw = sns.load_dataset('titanic')
print(f'Raw shape: {df_raw.shape}  |  Missing: {df_raw.isnull().sum().sum()}')
df_raw[['age', 'fare', 'sex', 'pclass', 'embarked', 'survived']].head()

## 1. Handle Missing Values

In [ ]:
print('Before:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

df = handle_missing(df_raw, strategy='median')
print(f'
After: {df.isnull().sum().sum()} missing values')

## 2. Encode Categoricals

In [ ]:
print('Cat columns before:', df.select_dtypes(include=['object','category']).columns.tolist())
print(f'Shape before: {df.shape}')

df_enc = encode_categoricals(df, method='onehot')
print(f'Shape after:  {df_enc.shape}')
new_cols = [c for c in df_enc.columns if c not in df.columns]
print('New columns:', new_cols)

## 3. Extract Date Features

In [ ]:
dates_df = pd.DataFrame({
    'date': pd.date_range('2020-01-01', periods=10, freq='15D').astype(str),
    'value': np.random.randn(10)
})
print('Before:')
print(dates_df.head(3))

dates_enc = create_date_features(dates_df, col='date')
print('
After date extraction:')
dates_enc.head(3)

## 4. Scale Numeric Features

In [ ]:
numeric_cols = ['age', 'fare']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[numeric_cols].boxplot(ax=axes[0])
axes[0].set_title('Before Scaling', fontweight='bold')

df_scaled = scale_features(df, cols=numeric_cols, method='standard')
df_scaled[numeric_cols].boxplot(ax=axes[1])
axes[1].set_title('After StandardScaler', fontweight='bold')

plt.suptitle('Effect of Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Polynomial & Interaction Features

In [ ]:
df_poly = add_polynomial_features(df[['age', 'fare']], cols=['age', 'fare'], degree=2)
print(f'Original: 2 features  →  After poly(degree=2): {df_poly.shape[1]} features')
print('New features:', [c for c in df_poly.columns if c not in ['age', 'fare']])
df_poly.head(3)

## 6. Full End-to-End Pipeline

In [ ]:
def preprocess_titanic(df_raw):
    df = df_raw.drop(columns=['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class'], errors='ignore')
    df = handle_missing(df, strategy='median')
    df = encode_categoricals(df, method='onehot')
    num_cols = df.select_dtypes(include='number').columns.drop('survived', errors='ignore').tolist()
    df = scale_features(df, cols=num_cols, method='robust')
    return df

df_ready = preprocess_titanic(df_raw)
print(f'Raw shape:   {df_raw.shape}')
print(f'Ready shape: {df_ready.shape}')
print(f'Missing:     {df_ready.isnull().sum().sum()}')
df_ready.head()

## Summary

| Step | Before | After |
|------|--------|-------|
| Missing values | ~20% in `age` | 0 |
| Categorical columns | 5 object cols | One-hot encoded |
| Numeric scale | `fare` up to 512 | Robust scaled |
| Feature count | 15 | ~22 after encoding |

➡️ **Next:** [03_ml_model_comparison.ipynb](03_ml_model_comparison.ipynb)